# 05 — SAR interpretation (which atoms drive predictions?)

Phase 6 of **CRC_Inhibitor_ML**. The trained Phase 4 multi-target model is a black box — it spits out predicted pIC50 numbers but doesn't tell us *why*. This notebook opens the box.

## Approach: vanilla gradient attribution

For each molecule + target pair, we:

1. Forward-pass through the model to get a predicted pIC50
2. Backward-pass the gradient of that prediction with respect to the atom feature matrix `x`
3. For each atom, take the magnitude of the gradient (summed across feature dimensions)
4. That magnitude = "how sensitive is the prediction to changes in this atom's features?" = "how much does the model 'lean on' this atom?"

Then we use RDKit's `SimilarityMaps` to render the molecule with each atom colored by its importance score. High-importance atoms appear darker red.

## Why this matters for the portfolio story

If the model is genuinely learning **structure-activity relationships** (and not just memorizing), the highlighted atoms should match known pharmacophores. For known EGFR inhibitors, we should see the model attending to the **quinazoline + aniline core** that medicinal chemists know is the EGFR-binding pharmacophore. If we see that, the model is doing real chemistry — not just pattern matching.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw
from rdkit.Chem.Draw import SimilarityMaps

from src.data.featurize import smiles_to_pyg
from src.data.proteins import get_target_embedding
from src.models.gine import load_multi_modal_gine

RDLogger.DisableLog("rdApp.*")

MODEL_PATH    = PROJECT_ROOT / "models" / "gine_esm2_multi_target.pt"
MAPPING_PATH  = PROJECT_ROOT / "data" / "raw" / "chembl_uniprot_mapping.txt"
ESM_CACHE     = PROJECT_ROOT / "data" / "processed" / "target_esm2_embeddings.pt"
FIGURES_DIR   = PROJECT_ROOT / "docs" / "figures" / "sar"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Figures will be saved to {FIGURES_DIR.relative_to(PROJECT_ROOT)}")

## Cell 1 — load the trained Phase 4 model

Same checkpoint the CLI tool uses. Loaded in eval mode.

In [ ]:
model, ckpt = load_multi_modal_gine(MODEL_PATH, device=device)
print(f"Model loaded. Trained on: {sorted(ckpt['targets'].values())}")
print(f"  atom_dim   = {ckpt['atom_dim']}")
print(f"  edge_dim   = {ckpt['edge_dim']}")
print(f"  target_dim = {ckpt['target_dim']}")

## Cell 2 — gradient attribution function

The core machinery: takes a SMILES + target embedding, returns one importance score per atom.

In [ ]:
def atom_attributions(model, smiles, target_emb, device):
    """Vanilla gradient attribution — per-atom importance via |dpred/dx|."""
    data = smiles_to_pyg(smiles, target_emb=target_emb, y=0.0)
    if data is None:
        return None, None
    data = data.to(device)
    # PyG needs a batch index even for a single graph
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
    data.x.requires_grad_(True)

    model.eval()
    pred = model(data)

    grad = torch.autograd.grad(pred.sum(), data.x, create_graph=False, retain_graph=False)[0]
    # Per-atom score: sum of |gradient| across feature dimensions
    importance = grad.abs().sum(dim=1).detach().cpu().numpy()
    return float(pred.item()), importance

## Cell 3 — visualization helper

Wraps RDKit's `SimilarityMaps.GetSimilarityMapFromWeights` to render a molecule with each atom shaded by its importance. Darker = the model thinks this atom matters more for the prediction.

In [ ]:
def plot_sar(smiles, importance, title=None, save_path=None):
    """Render molecule with atoms colored by importance score."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Could not parse SMILES: {smiles}")
        return

    # Normalize importance to [0, 1]
    weights = np.array(importance, dtype=float)
    if weights.max() > 0:
        weights = weights / weights.max()
    weights = weights.tolist()

    fig = SimilarityMaps.GetSimilarityMapFromWeights(
        mol, weights, colorMap="RdBu_r", size=(500, 500), contourLines=10,
    )
    if title:
        fig.suptitle(title, fontsize=12)
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path.relative_to(PROJECT_ROOT)}")
    plt.show()

## Cell 4 — attribute known EGFR inhibitors (CHEMBL203)

Take three FDA-approved EGFR drugs. For a model that has learned real chemistry, we should see attention concentrated on the **quinazoline ring system** (the heterocyclic core that binds the EGFR ATP pocket) and the **aniline / linker region** (which the medicinal chemistry literature has spent 25 years optimizing).

In [ ]:
egfr_emb = get_target_embedding(
    target="CHEMBL203",
    mapping_path=MAPPING_PATH,
    cache_path=ESM_CACHE,
    device=device,
)

egfr_compounds = [
    ("gefitinib", "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1"),
    ("erlotinib", "COc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC"),
    ("lapatinib", "CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1"),
]

for name, smi in egfr_compounds:
    pred, importance = atom_attributions(model, smi, egfr_emb, device)
    print(f"\n{name} vs EGFR: predicted pIC50 = {pred:.2f}")
    plot_sar(
        smi, importance,
        title=f"{name} vs EGFR  (pred pIC50 = {pred:.2f})",
        save_path=FIGURES_DIR / f"{name}_vs_egfr.png",
    )

## Cell 5 — same molecule, different target (target-awareness check)

Run gefitinib (an EGFR drug) against BRAF (a different kinase). The model should produce a *lower* predicted potency and the importance pattern should shift — because gefitinib doesn't bind BRAF the same way it binds EGFR. If the heatmap looks identical regardless of target, the model isn't actually target-aware.

In [ ]:
braf_emb = get_target_embedding(
    target="CHEMBL5145",
    mapping_path=MAPPING_PATH,
    cache_path=ESM_CACHE,
    device=device,
)

gefitinib = "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1"

for name, target_emb in [("EGFR", egfr_emb), ("BRAF", braf_emb)]:
    pred, importance = atom_attributions(model, gefitinib, target_emb, device)
    print(f"\ngefitinib vs {name}: predicted pIC50 = {pred:.2f}")
    plot_sar(
        gefitinib, importance,
        title=f"gefitinib vs {name}  (pred pIC50 = {pred:.2f})",
        save_path=FIGURES_DIR / f"gefitinib_vs_{name.lower()}.png",
    )

## Cell 6 — BRAF-specific compound (sorafenib)

Sorafenib is a clinically-approved BRAF inhibitor. The model should attend to its **diaryl urea** pharmacophore (the N-H-C(=O)-N-H bridge between two aryl rings — the canonical Type II kinase-binding motif). Verifying this is consistent with published medicinal chemistry of BRAF.

In [ ]:
sorafenib = "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1"

pred, importance = atom_attributions(model, sorafenib, braf_emb, device)
print(f"\nsorafenib vs BRAF: predicted pIC50 = {pred:.2f}")
plot_sar(
    sorafenib, importance,
    title=f"sorafenib vs BRAF  (pred pIC50 = {pred:.2f})",
    save_path=FIGURES_DIR / "sorafenib_vs_braf.png",
)

## Cell 7 — what to look for in the writeup

Inspect the saved figures (in `docs/figures/sar/`). For the writeup, the strongest evidence is:

1. **gefitinib vs EGFR** highlights atoms in the quinazoline ring + the aniline NH — both literature-confirmed pharmacophore regions for EGFR binding
2. **sorafenib vs BRAF** highlights atoms in the diaryl urea bridge — the textbook Type II kinase pharmacophore
3. **gefitinib vs BRAF** shows a *different* attention pattern than gefitinib vs EGFR (or scrambled / muted importance) — confirming target-awareness

If the heatmaps line up with these known pharmacophores, the model has genuinely learned chemistry. That's the headline finding for Phase 6. Even if R² didn't fully beat the random forest baseline, an interpretable model that correctly identifies known pharmacophores is meaningful work.